# §11.01 Graph Basics — Exercises

Eight graded exercises covering the core definitions, theorems, and algorithms
from the notes. Difficulty: ★ basic · ★★ intermediate · ★★★ advanced.

Run the **Setup** cell first.


In [ ]:
import numpy as np
import collections, math, itertools

def header(title, stars):
    print("=" * 60)
    print(f"  {title}  [{stars}]")
    print("=" * 60)

def check_close(name, got, expected, tol=1e-9):
    ok = abs(got - expected) <= tol
    print(f"  {'✓' if ok else '✗'} {name}: got {got}, expected {expected}")
    return ok

def check_true(name, condition, detail=""):
    print(f"  {'✓' if condition else '✗'} {name}{': ' + detail if detail else ''}")
    return condition

print("Setup complete.")


---
## Exercise 1 ★ — Handshaking Lemma

**Given** the edge list below, construct the adjacency matrix and verify the
handshaking lemma: $\sum_v \deg(v) = 2|E|$.

Also compute: (a) degree sequence, (b) max degree $\Delta$, (c) min degree $\delta$.

```
Vertices: {0, 1, 2, 3, 4}
Edges:    {0,1}, {0,2}, {1,2}, {1,3}, {2,4}, {3,4}
```


In [ ]:
# ── Scaffold ──────────────────────────────────────────────────────────────
edges = [(0,1),(0,2),(1,2),(1,3),(2,4),(3,4)]
n = 5

# TODO: build adjacency matrix
A = np.zeros((n, n), dtype=int)
# ... your code here ...

# TODO: compute degrees
degrees = None  # replace with row sums of A

# TODO: verify handshaking lemma
degree_sum = None
two_m      = None


In [ ]:
# ── Solution ──────────────────────────────────────────────────────────────
header("Exercise 1: Handshaking Lemma", "★")
edges = [(0,1),(0,2),(1,2),(1,3),(2,4),(3,4)]
n = 5
A = np.zeros((n, n), dtype=int)
for u, v in edges:
    A[u, v] = A[v, u] = 1

degrees     = A.sum(axis=1)
degree_sum  = int(degrees.sum())
two_m       = 2 * len(edges)
deg_seq     = sorted(degrees.tolist(), reverse=True)
Delta       = int(degrees.max())
delta       = int(degrees.min())

check_true("Adjacency matrix symmetric",  (A == A.T).all())
check_close("∑deg = 2|E|", degree_sum, two_m)
check_true("Degree sequence", deg_seq == [3, 3, 2, 2, 2],
           f"got {deg_seq}")
check_close("Δ(G)", Delta, 3)
check_close("δ(G)", delta, 2)
print()
print("Takeaway: every edge contributes exactly 1 to two vertex degrees,")
print("so the degree sum is always even and equals 2|E|.")


---
## Exercise 2 ★ — Walk Counting via Matrix Powers

Using the adjacency matrix from Exercise 1:

(a) Compute the number of walks of length 2 from vertex 0 to vertex 4.
(b) Compute the number of walks of length 3 from vertex 1 to vertex 1 (closed walks).
(c) Compute the total number of triangles in the graph using $\operatorname{tr}(A^3)/6$.


In [ ]:
# ── Scaffold ──────────────────────────────────────────────────────────────
edges = [(0,1),(0,2),(1,2),(1,3),(2,4),(3,4)]
n = 5
A = np.zeros((n, n), dtype=int)
for u, v in edges:
    A[u, v] = A[v, u] = 1

# TODO: compute A^2 and A^3
A2 = None
A3 = None

walks_0_to_4_len2 = None  # A2[0, 4]
closed_walks_1_len3 = None  # A3[1, 1]
n_triangles = None  # tr(A3) // 6


In [ ]:
# ── Solution ──────────────────────────────────────────────────────────────
header("Exercise 2: Walk Counting via Matrix Powers", "★")
edges = [(0,1),(0,2),(1,2),(1,3),(2,4),(3,4)]
n = 5
A = np.zeros((n, n), dtype=int)
for u, v in edges:
    A[u, v] = A[v, u] = 1

A2 = np.linalg.matrix_power(A, 2)
A3 = np.linalg.matrix_power(A, 3)

walks_0_to_4_len2  = int(A2[0, 4])
closed_walks_1_len3 = int(A3[1, 1])
n_triangles        = int(np.trace(A3)) // 6

check_close("Walks of length 2 from 0 to 4", walks_0_to_4_len2, 1)
check_close("Closed walks of length 3 at vertex 1", closed_walks_1_len3, 4)
check_close("Number of triangles", n_triangles, 1)
print()
print("Takeaway: (A^k)_{ij} counts all walks of length k — including those")
print("that revisit vertices. Diagonal entries count closed walks.")


---
## Exercise 3 ★ — Bipartiteness Detection

Implement BFS 2-colouring to test whether a graph is bipartite.
Test on:

- $C_6$ (should be bipartite)
- $C_5$ (should NOT be bipartite)
- $K_{3,3}$ (should be bipartite)

Return `(True, colouring_dict)` or `(False, None)`.


In [ ]:
# ── Scaffold ──────────────────────────────────────────────────────────────
def is_bipartite(adj_list, n):
    """
    adj_list: dict {v: [neighbours]}
    Returns (True, {v: 0/1}) or (False, None)
    """
    colour = {}
    # TODO: implement BFS 2-colouring
    return True, colour  # placeholder

# Test graphs (adjacency list format)
C6  = {i: [(i-1)%6, (i+1)%6] for i in range(6)}
C5  = {i: [(i-1)%5, (i+1)%5] for i in range(5)}
K33 = {i: [3,4,5] for i in range(3)}
K33.update({i: [0,1,2] for i in range(3,6)})


In [ ]:
# ── Solution ──────────────────────────────────────────────────────────────
header("Exercise 3: Bipartiteness Detection", "★")

def is_bipartite(adj_list, n):
    colour = {}
    for start in range(n):
        if start in colour: continue
        colour[start] = 0
        queue = collections.deque([start])
        while queue:
            v = queue.popleft()
            for u in adj_list.get(v, []):
                if u not in colour:
                    colour[u] = 1 - colour[v]
                    queue.append(u)
                elif colour[u] == colour[v]:
                    return False, None
    return True, colour

C6  = {i: [(i-1)%6, (i+1)%6] for i in range(6)}
C5  = {i: [(i-1)%5, (i+1)%5] for i in range(5)}
K33 = {i: [3,4,5] for i in range(3)}
K33.update({i: [0,1,2] for i in range(3,6)})

bip_c6,  col_c6  = is_bipartite(C6,  6)
bip_c5,  col_c5  = is_bipartite(C5,  5)
bip_k33, col_k33 = is_bipartite(K33, 6)

check_true("C_6 is bipartite",       bip_c6)
check_true("C_5 is NOT bipartite",   not bip_c5)
check_true("K_{3,3} is bipartite",   bip_k33)
if bip_c6:
    parts = ([v for v,c in col_c6.items() if c==0],
             [v for v,c in col_c6.items() if c==1])
    check_true("C_6 parts are equal size", len(parts[0])==len(parts[1]),
               f"parts={parts}")
print()
print("Takeaway: bipartite ↔ no odd cycle ↔ 2-colourable.")
print("BFS finds the 2-colouring or a contradicting odd cycle in O(n+m).")


---
## Exercise 4 ★★ — Spanning Trees and Kirchhoff's Theorem

(a) Implement a function that counts spanning trees of a graph using
**Kirchhoff's Matrix Tree Theorem**: the count equals any cofactor of the
Laplacian $L = D - A$ (i.e., $\det(L_{00})$ where $L_{00}$ is $L$ with row 0
and column 0 deleted).

(b) Verify Cayley's formula: $K_n$ should have $n^{n-2}$ spanning trees for
$n = 2, 3, 4, 5$.


In [ ]:
# ── Scaffold ──────────────────────────────────────────────────────────────
def count_spanning_trees(A):
    """
    Count spanning trees using Kirchhoff's Matrix Tree Theorem.
    A: numpy integer adjacency matrix (symmetric, zero diagonal).
    Returns: integer count.
    """
    # TODO: compute L, delete row/col 0, return rounded determinant
    pass

def complete_graph(n):
    return np.ones((n,n),dtype=int) - np.eye(n,dtype=int)


In [ ]:
# ── Solution ──────────────────────────────────────────────────────────────
header("Exercise 4: Spanning Trees & Kirchhoff", "★★")

def count_spanning_trees(A):
    D = np.diag(A.sum(axis=1))
    L = D - A
    L_sub = L[1:, 1:]
    return int(round(np.linalg.det(L_sub)))

def complete_graph(n):
    return np.ones((n,n),dtype=int) - np.eye(n,dtype=int)

all_ok = True
for n in [2, 3, 4, 5]:
    A = complete_graph(n)
    kirchhoff = count_spanning_trees(A)
    cayley    = n ** (n - 2)
    ok = check_close(f"K_{n}: spanning trees", kirchhoff, cayley)
    all_ok = all_ok and ok

# Also test a known graph: path P_4 has exactly 1 spanning tree (itself)
A_p4 = np.zeros((4,4),int)
for i in range(3): A_p4[i,i+1]=A_p4[i+1,i]=1
check_close("P_4: spanning trees", count_spanning_trees(A_p4), 1)
print()
print("Takeaway: Kirchhoff's theorem elegantly connects combinatorics (counting")
print("trees) to linear algebra (matrix determinants).")


---
## Exercise 5 ★★ — Connected Components and Bridges

(a) Implement BFS to find all connected components of a graph.

(b) Implement Tarjan's bridge-finding algorithm to find all **bridges** (edges
whose removal increases the number of components).

Test on a graph that has at least one bridge and at least two components.


In [ ]:
# ── Scaffold ──────────────────────────────────────────────────────────────
def connected_components(adj, n):
    """Return list of components (each a list of vertex indices)."""
    # TODO
    pass

def find_bridges(adj, n):
    """Return list of bridge edges (u, v)."""
    # TODO: Tarjan's algorithm
    pass

# Test graph: two triangles connected by one bridge
# Vertices 0-2: triangle; vertices 3-5: triangle; edge (2,3) is the bridge
test_adj = {
    0:[1,2], 1:[0,2], 2:[0,1,3],
    3:[2,4,5], 4:[3,5], 5:[3,4]
}


In [ ]:
# ── Solution ──────────────────────────────────────────────────────────────
header("Exercise 5: Components and Bridges", "★★")

def connected_components(adj, n):
    visited = [False]*n
    comps = []
    for start in range(n):
        if visited[start]: continue
        comp = []
        queue = collections.deque([start])
        visited[start] = True
        while queue:
            v = queue.popleft()
            comp.append(v)
            for u in adj.get(v, []):
                if not visited[u]:
                    visited[u] = True
                    queue.append(u)
        comps.append(sorted(comp))
    return comps

def find_bridges(adj, n):
    visited = [False]*n
    disc = [0]*n; low = [0]*n; timer = [0]
    bridges = []
    def dfs(u, parent):
        visited[u] = True
        disc[u] = low[u] = timer[0]; timer[0] += 1
        for v in adj.get(u, []):
            if not visited[v]:
                dfs(v, u)
                low[u] = min(low[u], low[v])
                if low[v] > disc[u]:
                    bridges.append((min(u,v), max(u,v)))
            elif v != parent:
                low[u] = min(low[u], disc[v])
    for i in range(n):
        if not visited[i]: dfs(i, -1)
    return sorted(bridges)

test_adj = {
    0:[1,2], 1:[0,2], 2:[0,1,3],
    3:[2,4,5], 4:[3,5], 5:[3,4]
}
n = 6
comps = connected_components(test_adj, n)
bridges = find_bridges(test_adj, n)

check_true("One connected component", len(comps)==1, f"got {comps}")
check_true("Bridge is edge (2,3)",    (2,3) in bridges, f"bridges={bridges}")
check_close("Exactly 1 bridge",       len(bridges), 1)
print()
print("Takeaway: bridges are single points of failure in networks.")
print("In ML, they correspond to critical edges in knowledge graphs or")
print("communication networks where message loss is most damaging.")


---
## Exercise 6 ★★ — Graph Coloring

(a) Implement greedy vertex colouring (largest-degree-first ordering).

(b) Verify that for the cycle $C_n$:
- $\chi(C_n) = 2$ when $n$ is even, $3$ when $n$ is odd.

(c) Using deletion-contraction, manually verify that
$P(C_3, k) = k(k-1)(k-2)$ by computing $P(P_3, k) - P(K_2, k)$.
Confirm $P(C_3, 3) = 6$.


In [ ]:
# ── Scaffold ──────────────────────────────────────────────────────────────
def greedy_color(adj, n):
    """
    Largest-degree-first greedy colouring.
    adj: dict {v: [neighbours]}
    Returns: dict {v: colour_index}
    """
    # TODO: sort vertices by degree descending, then assign smallest available colour
    pass

def cycle_adj(n):
    return {i: [(i-1)%n, (i+1)%n] for i in range(n)}


In [ ]:
# ── Solution ──────────────────────────────────────────────────────────────
header("Exercise 6: Graph Coloring", "★★")

def greedy_color(adj, n):
    order = sorted(range(n), key=lambda v: -len(adj.get(v,[])))
    colour = {}
    for v in order:
        used = {colour[u] for u in adj.get(v,[]) if u in colour}
        c = 0
        while c in used: c += 1
        colour[v] = c
    return colour

def cycle_adj(n):
    return {i: [(i-1)%n, (i+1)%n] for i in range(n)}

# Part (b): chromatic number of cycles
for n in [4, 5, 6, 7]:
    adj = cycle_adj(n)
    col = greedy_color(adj, n)
    chi = max(col.values()) + 1
    expected = 2 if n % 2 == 0 else 3
    check_close(f"χ(C_{n})", chi, expected)

# Part (c): deletion-contraction for C_3
# P(P_3, k) = k(k-1)^2;  P(K_2, k) = k(k-1)
# P(C_3, k) = P(P_3, k) - P(K_2, k) = k(k-1)^2 - k(k-1) = k(k-1)(k-2)
def P_path(n, k):
    return k * (k-1)**(n-1)

def P_complete(n, k):
    result = 1
    for i in range(n):
        result *= (k - i)
    return result

k = 3
P_C3 = P_path(3, k) - P_complete(2, k)
check_close("P(C_3, 3) = 6", P_C3, 6)
print()
print("Takeaway: deletion-contraction recursively reduces chromatic polynomial")
print("computation to smaller graphs. χ(G) is the smallest positive root.")


---
## Exercise 7 ★★★ — Weisfeiler-Leman Indistinguishability

The 1-WL test assigns iteratively refined colour labels to vertices.

(a) Implement 1-WL colour refinement.

(b) Show that 1-WL **cannot distinguish** $C_6$ from $2 \times K_3$ (two
disjoint triangles). Both graphs are 2-regular on 6 vertices; demonstrate
that their WL colour sequences converge to identical multisets at every round.

(c) Explain (in a comment) why this implies that a message-passing GNN cannot
distinguish these graphs either.


In [ ]:
# ── Scaffold ──────────────────────────────────────────────────────────────
def wl_colours(adj, n, n_rounds=6):
    """
    1-WL colour refinement.
    adj: dict {v: [neighbours]}
    Returns list of colour multisets per round: [Counter, Counter, ...]
    """
    # Initial colour = degree
    colours = {v: len(adj.get(v,[])) for v in range(n)}
    history = [collections.Counter(colours.values())]
    # TODO: iterate refinement
    return history

# C_6 and 2xK_3 adjacency lists
C6  = {i: [(i-1)%6, (i+1)%6] for i in range(6)}
K3K3 = {0:[1,2], 1:[0,2], 2:[0,1], 3:[4,5], 4:[3,5], 5:[3,4]}


In [ ]:
# ── Solution ──────────────────────────────────────────────────────────────
header("Exercise 7: 1-WL Indistinguishability", "★★★")

def wl_colours(adj, n, n_rounds=6):
    colours = {v: len(adj.get(v,[])) for v in range(n)}
    history = [collections.Counter(colours.values())]
    for _ in range(n_rounds):
        new_c = {}
        for v in range(n):
            nb_sig = tuple(sorted(colours[u] for u in adj.get(v,[])))
            new_c[v] = hash((colours[v], nb_sig)) % (10**9)
        # Normalise to small integers
        unique = sorted(set(new_c.values()))
        remap = {c: i for i, c in enumerate(unique)}
        colours = {v: remap[new_c[v]] for v in range(n)}
        h = collections.Counter(colours.values())
        history.append(h)
        if history[-1] == history[-2]:
            break
    return history

C6   = {i: [(i-1)%6, (i+1)%6] for i in range(6)}
K3K3 = {0:[1,2], 1:[0,2], 2:[0,1], 3:[4,5], 4:[3,5], 5:[3,4]}

h_c6   = wl_colours(C6,   6)
h_k3k3 = wl_colours(K3K3, 6)

print("Round-by-round colour multisets:")
for r, (c6, k3) in enumerate(zip(h_c6, h_k3k3)):
    same = c6 == k3
    print(f"  Round {r}: C_6={dict(c6)}  2xK_3={dict(k3)}  same={same}")

all_same = all(c6 == k3 for c6, k3 in zip(h_c6, h_k3k3))
check_true("All rounds: WL cannot distinguish C_6 from 2xK_3", all_same)
print()
print("# WHY THIS MATTERS FOR GNNs:")
print("# A message-passing GNN with sum/mean/max aggregation is at most as")
print("# powerful as 1-WL (Xu et al., ICLR 2019). Since 1-WL produces")
print("# identical colour sequences for C_6 and 2xK_3, any such GNN will")
print("# output the same graph-level representation for both — even though")
print("# they are structurally different (one is connected, the other is not).")
print("# This fundamental limitation motivates higher-order GNNs and graph")
print("# transformers that go beyond the 1-WL expressiveness ceiling.")


---
## Exercise 8 ★★★ — Planarity, Euler's Formula, and Kuratowski's Theorem

(a) Verify Euler's formula $n - m + f = 2$ for the following planar graphs by
computing the number of faces $f$ from $n$ and $m$:

| Graph | $n$ | $m$ |
|-------|-----|-----|
| $K_4$ | 4 | 6 |
| $C_6$ | 6 | 6 |
| Octahedron | 6 | 12 |
| Cube ($Q_3$) | 8 | 12 |

(b) Show that $K_5$ and $K_{3,3}$ **cannot be planar** by checking that they
violate the necessary density bound $m \leq 3n - 6$ (and for bipartite graphs,
$m \leq 2n - 4$).

(c) For the house graph ($n=5$, $m=6$), compute $f$ and describe all faces.


In [ ]:
# ── Scaffold ──────────────────────────────────────────────────────────────
def euler_faces(n, m):
    """Return number of faces from Euler's formula: f = 2 - n + m."""
    # TODO
    pass

def is_planar_by_density(n, m, bipartite=False):
    """
    Check necessary density condition for planarity.
    For general graphs: m <= 3n - 6
    For bipartite:      m <= 2n - 4
    Returns (could_be_planar: bool, reason: str)
    """
    # TODO
    pass


In [ ]:
# ── Solution ──────────────────────────────────────────────────────────────
header("Exercise 8: Planarity & Euler's Formula", "★★★")

def euler_faces(n, m):
    return 2 - n + m

def is_planar_density(n, m, bipartite=False):
    bound = (2*n - 4) if bipartite else (3*n - 6)
    label = "2n-4" if bipartite else "3n-6"
    ok = m <= bound
    return ok, f"m={m} {'<=' if ok else '>'} {label}={bound}"

# Part (a): Euler's formula
planar_graphs = [
    ("K_4",      4,  6, 4),   # 4 faces: 4 triangular + outer
    ("C_6",      6,  6, 2),   # 2 faces: interior + exterior
    ("Octahedron",6, 12, 8),  # 8 triangular faces
    ("Cube Q_3", 8, 12, 6),   # 6 square faces
]
print("Part (a): Euler's formula f = 2 - n + m")
for name, n, m, f_expected in planar_graphs:
    f = euler_faces(n, m)
    check_close(f"f({name})", f, f_expected)

# Part (b): density bounds
print("\nPart (b): Density bounds for non-planarity")
non_planar = [
    ("K_5",     5, 10, False),
    ("K_{3,3}", 6,  9, True),   # bipartite
]
for name, n, m, bip in non_planar:
    ok, reason = is_planar_density(n, m, bipartite=bip)
    check_true(f"{name} fails density bound (non-planar)", not ok, reason)

# Part (c): house graph
print("\nPart (c): House graph faces")
n_h, m_h = 5, 6
f_h = euler_faces(n_h, m_h)
check_close("House: f = 2 - 5 + 6 = 3", f_h, 3)
print("  Faces: (1) outer/infinite face, (2) bottom rectangle {0,1,2,3},")
print("         (3) roof triangle {2,3,4}.")
print()
print("Takeaway: Euler's formula is a topological invariant — it holds for")
print("ANY planar embedding of a connected planar graph, regardless of shape.")
print("Violation of the density bound gives an easy planarity certificate.")
